In [1]:
import pandas as pd
import os
from pathlib import Path
import pandas as pd
from typing import List, Tuple
import re

## Merging Data

In [71]:
ROOT_FOLDER = Path(r"C:\Users\Microsoft\OneDrive\Desktop\Assignment3\compressed-attachments")
OUTPUT_FILE = "travel_dataset.csv"

In [72]:
COLUMN_MAPPING = {
    'image_url': ['image url', 'image_url', 'imageurl', 'url', 'image', 'picture', 'link','image_url_description'],
    'description': ['description', 'describtion'],
    'country': ['country', 'destination name','destination_name'],
    'weather': ['weather'],
    'time_of_day': ['time of day', 'time_of_day', 'timeofday'],
    'season': ['season'],
    'activity': ['activity'],
    'mood': ['mood/emotion', 'mood_emotion', 'mood', 'emotion', 'mode','moodemotion']
}


### Normalize col names 

In [73]:
def normalize_column_name(col: str) -> str:
    col = col.replace('ï»¿', '')
    col = re.sub(r'[:\s]+$', '', col)
    col = re.sub(r'^[\s]+', '', col)
    col = col.lower().strip()
    col = re.sub(r'[\s_]+', '_', col)
    col = re.sub(r'[^\w]', '', col)
    
    return col


In [ ]:
def MapToStandards(df: pd.DataFrame):
    normalized_cols = {col: normalize_column_name(col) for col in df.columns}
    df = df.rename(columns=normalized_cols)
    reverse_mapping = {}
    for standard_name, variations in COLUMN_MAPPING.items():
        for variation in variations:
            reverse_mapping[variation] = standard_name
    
    rename_dict = {}
    for col in df.columns:
        if col in reverse_mapping:
            rename_dict[col] = reverse_mapping[col]
    df = df.rename(columns=rename_dict)
    df = df.loc[:, ~df.columns.str.contains('^unnamed', case=False, na=False)]
    df = df.loc[:, ~df.columns.duplicated()]
    
    return df


In [6]:
def read_csv_safely(file_path: Path) -> Tuple[pd.DataFrame | None, str | None]:
    try:
        df = pd.read_csv(file_path, encoding="latin1", engine="python", on_bad_lines="skip")
        
        # Normalize columns
        df = MapToStandards(df)
        if df.shape[1] < 1:
            return None, "no valid columns after normalization"
        
        return df, None
    
    except Exception as e:
        return None, str(e)

In [7]:
def collect_csv_files(root_folder: Path) -> List[Path]:
    return list(root_folder.rglob("*.csv"))

In [ ]:
def merge_travel_datasets(root_folder: Path, output_file: str) -> None:
    csv_files = collect_csv_files(root_folder)
    print(f"Found {len(csv_files)} CSV files\n")
    valid_dfs = []
    skipped = []
    
    for file_path in csv_files:
        print(f"Reading: {file_path.name}")
        df, error = read_csv_safely(file_path)
        
        if df is not None:
            valid_dfs.append(df)
            print(f"  Columns: {list(df.columns)}")
        else:
            print(f"   Skipped: {error}")
            skipped.append((file_path.name, error))
    
    if not valid_dfs:
        print("\nNo valid CSV files found!")
        return
    merged_df = pd.concat(valid_dfs, ignore_index=True, sort=False)
    standard_cols = list(COLUMN_MAPPING.keys())
    existing_standard = [col for col in standard_cols if col in merged_df.columns]
    other_cols = [col for col in merged_df.columns if col not in existing_standard]
    merged_df = merged_df[existing_standard + other_cols]

    merged_df.drop_duplicates(inplace=True)
    
    merged_df.to_csv(output_file, index=False, encoding="utf-8")
    
    print(f"\n✓ DATASET CREATED: {output_file}")
    print(f"  - Total files processed: {len(csv_files)}")
    print(f"  - Successfully merged: {len(valid_dfs)}")
    print(f"  - Skipped: {len(skipped)}")
    print(f"  - Final rows: {len(merged_df):,}")
    print(f"  - Final columns: {merged_df.shape[1]}")
    print(f"  - Column names: {list(merged_df.columns)}")

In [9]:
merge_travel_datasets(ROOT_FOLDER, OUTPUT_FILE)

Found 121 CSV files

Reading: 2936035-1161937.csv
  ✓ Columns: ['image_url', 'description', 'country', 'weather', 'time_of_day', 'season', 'activity', 'mood']
Reading: 2938271-1181804.csv
  ✓ Columns: ['image_url', 'description', 'country', 'weather', 'time_of_day', 'season', 'activity', 'mood']
Reading: 2937999-1190855.csv
  ✓ Columns: ['image_url', 'description', 'country', 'weather', 'time_of_day', 'season', 'activity', 'mood']
Reading: 2938082-1190855.csv
  ✓ Columns: ['image_url', 'description', 'country', 'weather', 'time_of_day', 'season', 'activity', 'mood']
Reading: 2938087-1200242.csv
  ✓ Columns: ['image_url', 'description', 'country', 'weather', 'time_of_day', 'season', 'activity', 'mood']
Reading: 2934979-1200431.csv
  ✓ Columns: ['image_url', 'description', 'country', 'weather', 'time_of_day', 'season', 'activity', 'mood']
Reading: 2938588-1200964 .csv
  ✓ Columns: ['image_url', 'description', 'country', 'weather', 'time_of_day', 'season', 'activity', 'mood']
Reading: 293

## Explore Dataset

In [74]:
df = pd.read_csv("travel_dataset.csv")

In [75]:
df.columns

Index(['image_url', 'description', 'country', 'weather', 'time_of_day',
       'season', 'activity', 'mood'],
      dtype='object')

In [76]:
columns = ['image_url', 'description', 'country', 'weather', 'time_of_day',
       'season', 'activity', 'mood']

In [13]:
df = df[columns]


In [14]:
df.to_csv("travel_dataset.csv",index=False)

In [15]:
df = pd.read_csv("travel_dataset.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

In [77]:
df.head()

,image_url,description,country,weather,time_of_day,season,activity,mood
0,https://commons.wikimedia.org/wiki/File:Dom_of...,A clear image of the Dome of the Rock in Jerus...,Palestine,Sunny,Afternoon,Summer,Sightseeing,Nostalgia
1,https://upload.wikimedia.org/wikipedia/commons...,a clear image of the Ibrahimi Mosque (Cave of ...,Palestine,Sunny,Morning,Spring,Sightseeing,Curiosity
2,https://upload.wikimedia.org/wikipedia/commons...,A clear image of the ancient ruins in Sebastia...,Palestine,Sunny,Afternoon,Summer,Exploring,Adventure
3,https://upload.wikimedia.org/wikipedia/commons...,A clear image of Mar Saba Monastery in Bethleh...,Palestine,Sunny,Afternoon,Summer,Sightseeing,Awe
4,https://upload.wikimedia.org/wikipedia/commons...,A clear aerial view of Tell es-Sultan in Jeric...,Palestine,Sunny,Morning,Spring,Exploring,Curiosity


### Data Cleaining

In [78]:
df.shape

(1457, 8)

In [79]:
df = df[df["image_url"].astype(str).str.match(r"^https?://")]

In [80]:
df["description"].isna().sum()

np.int64(10)

In [81]:
df["description"] = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [82]:
df.sample(5)

,image_url,description,country,weather,time_of_day,season,activity,mood
91,https://www.ricksteves.com/watch-read-listen/r...,Swiss cows grazing in the Berner Oberland Alps...,Switzerland,Sunny,Afternoon,Summer,Viewing nature / countryside,Relaxing
876,https://cdn-imgix.headout.com/media/images/ab8...,Cappadociaâs valleys filled with hot-air bal...,TÃ¼rkiye,Sunny,Morning,Spring,Hot-air balloon ride,Adventure
1176,https://i.postimg.cc/m2t5SJ2V/image3.png,"A rugged mountainous landscape in Afghanistan,...",Afghanistan,Sunny,Afternoon,Summer,Hiking,Adventure
197,https://www.fairflight.de/blog/wp-content/uplo...,A bright coastal view of Waikiki Beach with tu...,USA,Sunny,Morning,Summer,Relaxing,Happiness
451,https://upload.wikimedia.org/wikipedia/commons...,"The red desert dunes of Djanet, Algeria  a st...",Algeria,Sunny,Afternoon,Not Clear,Exploring,Adventure


In [83]:
df.isna().sum()

image_url       0
description     0
country         1
weather         0
time_of_day     0
season          0
activity       22
mood            1
dtype: int64

In [84]:
df["activity"] = df["activity"].fillna("Unknown")

In [85]:
df = df[df["country"].notna()]
df = df[df["mood"].notna()]

In [86]:
df["season"] = (
    df["season"]
    .fillna("")
    .astype(str)
    .str.replace("\t", " ", regex=False)
    .str.strip()
    .str.lower()
)


In [87]:
df["season"].value_counts()

season
summer       407
spring       196
winter       159
fall         136
not clear    119
autumn         9
clear          1
springl        1
Name: count, dtype: int64

In [88]:
df[df["season"] == "autumn"] = "fall"

In [89]:
df[df["season"] == "springl"] = "spring"
df = df[df["season"] != "clear"]

In [90]:
df["season"].value_counts()

season
summer       407
spring       197
winter       159
fall         145
not clear    119
Name: count, dtype: int64

In [93]:
df["weather"].value_counts()

weather
sunny                         550
cloudy                        204
not clear                      89
snowy                          87
clear                          45
rainy                          21
fall                            9
clear / sunny                   5
windy                           3
partly cloudy                   2
cold                            2
raniy                           2
not clear (night lighting)      1
partly cloudy / sunny           1
cloudt                          1
cold / clear                    1
spring                          1
clear night sky                 1
cloudy sunset                   1
clear night                     1
Name: count, dtype: int64

In [92]:
df["weather"] = (
    df["weather"]
    .fillna("")
    .astype(str)
    .str.replace("\t", " ", regex=False)
    .str.replace("\n", " ", regex=False)
    .str.strip()
    .str.lower()
)


In [ ]:
def normalize_weather(w):
    if "sunny" in w and not any(x in w for x in ["cloud", "rain", "snow"]):
        return "Sunny"
    if "cloud" in w and not any(x in w for x in ["rain", "snow"]):
        return "Cloudy"
    if "rain" in w:
        return "Rainy"
    if "snow" in w:
        return "Snowy"
    if "not clear" in w:
        return "Not Clear"
    return None


In [95]:
df["weather_clean"] = df["weather"].apply(normalize_weather)

In [96]:
df = df[df["weather_clean"].notna()]
df["weather"] = df["weather_clean"]
df = df.drop(columns=["weather_clean"])


In [97]:
df["weather"].value_counts()

weather
Sunny        555
Cloudy       209
Not Clear     90
Snowy         87
Rainy         21
Name: count, dtype: int64

In [100]:
df["time_of_day"].value_counts()

time_of_day
afternoon              459
evening                251
morning                228
night                    7
not clear                7
daytime                  5
sunset                   2
morning,                 2
afternoon afternoon      1
Name: count, dtype: int64

In [99]:
df["time_of_day"] = (
    df["time_of_day"]
    .fillna("")
    .astype(str)
    .str.replace("\t", " ", regex=False)
    .str.replace("\n", " ", regex=False)
    .str.strip()
    .str.lower()
)


In [101]:
def normalize_time(t):
    if "morning" in t:
        return "Morning"

    if "afternoon" in t:
        return "Afternoon"

    if "evening" in t:
        return "Evening"

    if "sunset" in t:
        return "Evening"

    if "night" in t:
        return "Evening"

    if "not clear" in t or "daytime" in t:
        return "Not Clear"

    return None


In [102]:
df["time_clean"] = df["time_of_day"].apply(normalize_time)

In [103]:
df = df[df["time_clean"].notna()]
df["time_of_day"] = df["time_clean"]
df = df.drop(columns=["time_clean"])

In [110]:
df["mood"].value_counts()

mood
happiness              198
adventure              174
curiosity              170
excitement             134
romance                112
nostalgia               93
melancholy              19
awe                      9
calm                     6
peace                    5
calmness                 4
wonder                   3
peacefulness             2
inspiration              2
romantic                 2
serenity                 2
reverence                2
advanture                2
curosity                 2
peaceful                 2
curiosty                 2
relaxing                 1
amazed                   1
exitment                 1
relaxation               1
impressed                1
awe and adventure        1
curiousity               1
adventurous              1
spirituality             1
excitement  passion      1
curiosity  awe           1
melanchol                1
reverence  awe           1
humility                 1
spiritual awe            1
tranquility            

In [ ]:
df["mood"] = (
    df["mood"]
    .fillna("")
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)  
    .str.strip()  
)

In [ ]:
mood_mapping = {
    'happiness': 'Happy',
    'joy': 'Happy',
    
    'excitement': 'Excited',
    'exitment': 'Excited',
    'excitement  passion': 'Excited',
    'adventure': 'Excited',
    'advanture': 'Excited',
    'adventurous': 'Excited',
    'awe and adventure': 'Excited',
    
    'curiosity': 'Curious',
    'curosity': 'Curious',
    'curiosty': 'Curious',
    'curiousity': 'Curious',
    'curiosity  awe': 'Curious',
    'awe': 'Curious',
    'wonder': 'Curious',
    'amazed': 'Curious',
    'impressed': 'Curious',
    
    'romance': 'Romantic',
    'romantic': 'Romantic',
    'nostalgia': 'Romantic',
    
    'calm': 'Calm',
    'peace': 'Calm',
    'calmness': 'Calm',
    'peacefulness': 'Calm',
    'peaceful': 'Calm',
    'serenity': 'Calm',
    'tranquility': 'Calm',
    'relaxation': 'Calm',
    'relaxing': 'Calm',
    'melancholy': 'Calm',  
    'melanchol': 'Calm',
    'reverence': 'Calm',
    'reverence  awe': 'Calm',
    'spirituality': 'Calm',
    'spiritual awe': 'Calm',
    'humility': 'Calm',
    'inspiration': 'Calm',
}

In [ ]:
df['mood_general'] = df['mood'].map(mood_mapping)

unmapped = df[df['mood_general'].isna()]['mood'].unique()
if len(unmapped) > 0:
    print(f"WARNING: Unmapped moods: {unmapped}")
else:
    print("All moods successfully mapped!")

print("\nFinal distribution:")
print(df['mood_general'].value_counts())
print("\nPercentages:")
print(df['mood_general'].value_counts(normalize=True).mul(100).round(1))

✅ All moods successfully mapped!

Final distribution:
mood_general
Excited     314
Romantic    207
Happy       199
Curious     190
Calm         52
Name: count, dtype: int64

Percentages:
mood_general
Excited     32.6
Romantic    21.5
Happy       20.7
Curious     19.8
Calm         5.4
Name: proportion, dtype: float64


In [114]:
df['mood'] = df['mood_general'].copy()
df.drop(columns=['mood_general'], inplace=True)

In [115]:
df["mood"].value_counts()

mood
Excited     314
Romantic    207
Happy       199
Curious     190
Calm         52
Name: count, dtype: int64

In [120]:
df["activity"].value_counts()

activity
sightseeing              449
relaxing                 146
hiking                    89
exploring                 23
unknown                   22
                        ... 
walking  photography       1
traveling                  1
hotair balloon riding      1
eating                     1
snorkelling                1
Name: count, Length: 141, dtype: int64

In [ ]:
df["activity"] = (
    df["activity"]
    .fillna("")
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)  
    .str.strip()  
)


In [121]:
def normalize_activity(a):
    if "sight" in a:
        return "Sightseeing"

    if a in ["relaxing", "relaxation", "chilling"]:
        return "Relaxing"

    if "hiking" in a or "trek" in a:
        return "Hiking"

    if "explor" in a:
        return "Exploring"

    if any(x in a for x in ["food", "eat", "dining", "restaurant"]):
        return "Food"

    if any(x in a for x in ["snorkel", "diving", "swimming"]):
        return "Water Activities"

    if any(x in a for x in ["balloon", "adventure", "ride"]):
        return "Adventure"

    if a == "unknown" or a == "":
        return "Unknown"

    return "Other"


In [122]:
df["activity"] = df["activity"].apply(normalize_activity)


In [123]:
df["activity"].value_counts()

activity
Sightseeing         482
Other               158
Relaxing            146
Hiking               94
Exploring            38
Unknown              22
Water Activities     12
Adventure             9
Food                  1
Name: count, dtype: int64

In [127]:
df["country"].value_counts()

country
italy          65
japan          55
switzerland    50
france         49
spain          48
               ..
bolivia         1
costa rica      1
serbia          1
colombia        1
zimbabwe        1
Name: count, Length: 131, dtype: int64

In [ ]:
df["country"] = (
    df["country"]
    .fillna("")
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)  
    .str.strip()  
)

In [129]:
country_map = {
    "uae": "United Arab Emirates",
    "u.a.e": "United Arab Emirates",
    "united arab emirates": "United Arab Emirates",

    "uk": "United Kingdom",
    "u.k": "United Kingdom",
    "united kingdom": "United Kingdom",

    "usa": "United States",
    "u.s.a": "United States",
    "united states": "United States",
    "united states of america": "United States",
}


In [130]:
df["country"] = df["country"].replace(country_map)


In [131]:
df["country"] = df["country"].str.title()


In [132]:
print(df["country"].value_counts().head(20))


country
Italy                   65
Japan                   55
Switzerland             50
France                  49
Spain                   48
Palestine               45
United States           42
Maldives                40
Greece                  36
Turkey                  33
Egypt                   29
Saudi Arabia            23
China                   22
United Arab Emirates    22
United Kingdom          20
Canada                  19
Jordan                  18
Iceland                 18
Brazil                  17
Austria                 16
Name: count, dtype: int64


In [133]:
df.shape

(962, 8)

In [ ]:
import requests

def is_image_url(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r = requests.head(url, timeout=5, allow_redirects=True, headers=headers)
        ctype = r.headers.get("Content-Type", "")
        if ctype.startswith("image"):
            return True
    except:
        pass
    try:
        r = requests.get(url, stream=True, timeout=5, headers=headers)
        ctype = r.headers.get("Content-Type", "")
        return ctype.startswith("image")
    except:
        return False


In [59]:
df["is_image"] = df["image_url"].apply(is_image_url)
print("Sure image URLs:", df["is_image"].sum())


Sure image URLs: 764


In [60]:
import requests
from PIL import Image
from io import BytesIO

def load_image_safe(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, timeout=10, headers=headers)
        r.raise_for_status()
        img = Image.open(BytesIO(r.content)).convert("RGB")
        return img
    except:
        return None


In [142]:
import re

def sanitize_url(url):
    if not isinstance(url, str):
        return None

    url = url.strip()

    # إصلاح التكرار إن وُجد
    url = re.sub(r'(\.(jpg|jpeg|png))\1+$', r'\1', url, flags=re.IGNORECASE)

    return url


In [144]:
df['image_url'] = df['image_url'].apply(sanitize_url)


In [61]:
valid = 0
for url in df["image_url"]:
    if load_image_safe(url) is not None:
        valid += 1

print("Loadable images:", valid)


Loadable images: 769


In [135]:
df["description"].value_counts()

description
intersting city                                                                                                                                    3
                                                                                                                                                   3
A magical winter view of Northern Lights. I chose this place because I dream of seeing the aurora in real life                                     3
quit place amd nice views                                                                                                                          3
i love to discover islamic contries and their hestory                                                                                              3
                                                                                                                                                  ..
A peaceful lakeside village surrounded by misty mountains, colorful alpine houses, and calm re

In [136]:
df["description"] = df["description"].fillna("").astype(str)
df["description"] = df["description"].str.replace("\ufeff", "", regex=False)  # BOM
df["description"] = df["description"].str.strip()
df = df[df["description"] != ""]


In [137]:
import re
df.loc[:, "description"] = df["description"].apply(
    lambda s: re.sub(r"\s+", " ", s).strip()
)



In [138]:
df.loc[:, "description"] = df["description"].str.strip(' "')


In [139]:
df = df.copy()

df.loc[:, "description"] = (
    df["description"]
    .astype(str)
    .str.replace("\ufeff", "", regex=False)   # remove BOM
    .str.strip()                               # trim whitespace
    .str.strip('"')                            # remove leading/trailing "
    .str.strip("'")                            # remove leading/trailing '
    .str.replace(r"\s+", " ", regex=True)     # normalize spaces
)


In [140]:
df.head(10)

,image_url,description,country,weather,time_of_day,season,activity,mood
0,https://commons.wikimedia.org/wiki/File:Dom_of...,A clear image of the Dome of the Rock in Jerus...,Palestine,Sunny,Afternoon,summer,Sightseeing,Romantic
1,https://upload.wikimedia.org/wikipedia/commons...,a clear image of the Ibrahimi Mosque (Cave of ...,Palestine,Sunny,Morning,spring,Sightseeing,Curious
2,https://upload.wikimedia.org/wikipedia/commons...,A clear image of the ancient ruins in Sebastia...,Palestine,Sunny,Afternoon,summer,Exploring,Excited
3,https://upload.wikimedia.org/wikipedia/commons...,A clear image of Mar Saba Monastery in Bethleh...,Palestine,Sunny,Afternoon,summer,Sightseeing,Curious
4,https://upload.wikimedia.org/wikipedia/commons...,A clear aerial view of Tell es-Sultan in Jeric...,Palestine,Sunny,Morning,spring,Exploring,Curious
5,https://upload.wikimedia.org/wikipedia/commons...,A night view of the Eiffel Tower illuminated i...,France,Not Clear,Afternoon,not clear,Sightseeing,Romantic
6,https://upload.wikimedia.org/wikipedia/commons...,A bright and clear view of the Great Wall of C...,China,Sunny,Morning,summer,Hiking,Excited
9,https://upload.wikimedia.org/wikipedia/commons...,A bright and peaceful beach view in the Maldiv...,Maldives,Sunny,Afternoon,summer,Relaxing,Happy
10,https://images.unsplash.com/photo-147843612789...,Walking through the endless vermilion torii ga...,Japan,Sunny,Morning,spring,Hiking,Romantic
11,https://images.unsplash.com/photo-157007718867...,White buildings with blue domes overlooking th...,Greece,Sunny,Afternoon,summer,Relaxing,Happy


In [145]:
df.to_csv("travel_dataset_clean_final.csv", index=False, encoding="utf-8")
print("Saved: travel_dataset_clean_final.csv")
print("Rows:", len(df))


Saved: travel_dataset_clean_final.csv
Rows: 959
